# 02. RFM analysis and customer segmentation

## 8. Customer-level feature engineering
Each customer becomes one row. Guests are excluded because they have no identity. Customers whose purchases were entirely returned are excluded too, because they have no net spend to score.

In [1]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd
import plotly.express as px, plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"
pio.templates.default = "plotly_white"
pd.set_option("display.max_columns", 40, "display.width", 180, "display.float_format", lambda v: f"{v:,.2f}")
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
OUT = ROOT / "data" / "processed"
sales = pd.read_parquet(OUT / "sales.parquet"); returns = pd.read_parquet(OUT / "returns.parquet")
rfm = pd.read_parquet(OUT / "rfm.parquet"); products = pd.read_parquet(OUT / "products.parquet")
invoices = pd.read_parquet(OUT / "invoices.parquet"); cohorts = pd.read_parquet(OUT / "cohorts.parquet")
gaps = pd.read_parquet(OUT / "gaps.parquet"); affinity = pd.read_parquet(OUT / "affinity.parquet")
audit = json.loads((OUT / "audit.json").read_text()); evals = json.loads((OUT / "cluster_eval.json").read_text())
sales["CustomerID"] = sales["CustomerID"].astype(str)
print(f"{len(sales):,} clean sale lines, {sales['Invoice'].nunique():,} invoices, {len(rfm):,} scored customers")

1,003,340 clean sale lines, 39,516 invoices, 5,839 scored customers


In [2]:
snapshot = pd.Timestamp(audit["snapshot_date"]); print("Scored as of", snapshot.date())
rfm[["CustomerID", "Recency", "Frequency", "Monetary", "AOV", "Products", "Tenure", "Country"]].head()

Scored as of 2011-12-10


,CustomerID,Recency,Frequency,Monetary,AOV,Products,Tenure,Country
0,12346,529,2,169.36,84.68,24,647,United Kingdom
1,12347,2,8,"4,921.53",615.19,126,404,Iceland
2,12348,75,5,"1,658.40",331.68,24,438,Finland
3,12349,18,3,"3,678.69","1,226.23",137,589,Italy
4,12350,310,1,294.40,294.40,16,310,Norway


## 9. RFM analysis
- **Recency** is days from the last purchase to the snapshot date (10 Dec 2011, the day after the data ends). **Lower is better**, so the score is reversed: a score of 5 means most recent.
- **Frequency** is the number of distinct invoices with at least one line not fully returned. Higher is better.
- **Monetary** is net revenue after returns. Higher is better.

Scores split customers into five equal groups with `pd.qcut`. Frequency is heavily tied, with many customers at one or two orders, so plain `qcut` cannot form five bins. Ranking first breaks the ties, which means customers with the same frequency can land in neighbouring scores. A score is therefore a rank within this customer base, not an absolute level.

In [3]:
display(rfm[["R", "F", "M"]].apply(pd.Series.value_counts).sort_index())
print(rfm.groupby("R").Recency.agg(["min", "max", "median"]))
print(rfm.groupby("F").Frequency.agg(["min", "max", "median"]))
print(rfm.groupby("M").Monetary.agg(["min", "max", "median"]))
rfm["RFM"].value_counts().head(10)

,R,F,M
1,1168,1168,1168
2,1168,1168,1168
3,1167,1167,1167
4,1168,1168,1168
5,1168,1168,1168


   min  max  median
R                  
1  408  738  531.50
2  186  408  309.00
3   58  186   95.00
4   19   58   33.00
5    0   19    8.00
   min  max  median
F                  
1    1    1    1.00
2    1    2    2.00
3    2    4    3.00
4    4    8    6.00
5    8  370   13.00
       min        max   median
M                             
1     2.90     278.34   159.05
2   278.55     593.19   397.00
3   593.50   1,180.23   844.86
4 1,180.47   2,844.61 1,747.24
5 2,846.13 578,408.64 5,251.09


RFM
555    463
111    314
455    246
121    169
211    167
344    160
112    148
444    145
355    134
122    132
Name: count, dtype: int64

## 10. RFM visualisation and interpretation
Segment rules, first match wins, so every customer gets exactly one segment:

| Segment | Rule | Reasoning |
|---|---|---|
| Champions | R=5, F>=4, M>=4 | The brief's 555 alone is too narrow, so this keeps the top on all three with some tolerance |
| Loyal customers | R>=4, F>=3, M>=3 | Recent and solid, without being top on everything |
| At-risk high value | R<=2, F>=3, M>=3 | Strong history, but gone quiet |
| Cooling regulars | R=3, F>=3, M>=3 | Added from the data: a large group between loyal and at-risk, and early action costs less |
| New inflow | R>=4, F<=2, M<=2 | Recent first purchases |
| Promising | R>=3, not above | Recent but modest |
| Lost one-timers | R=1, F<=2 | Long gone with little history |
| Hibernating | Everyone else | Old and low value |

In [4]:
seg = rfm.groupby("Segment").agg(Customers=("CustomerID", "size"), Revenue=("Monetary", "sum"), MedianR=("Recency", "median"), MedianF=("Frequency", "median"), MedianM=("Monetary", "median"))
seg["CustomerShare"] = seg.Customers / seg.Customers.sum(); seg["RevenueShare"] = seg.Revenue / seg.Revenue.sum()
seg = seg.sort_values("Revenue", ascending=False); display(seg)
long = seg.reset_index().melt(id_vars="Segment", value_vars=["CustomerShare", "RevenueShare"])
px.bar(long, y="Segment", x="value", color="variable", barmode="group", orientation="h", title="Share of customers against share of revenue").show()
px.imshow(rfm.pivot_table(index="R", columns="F", values="CustomerID", aggfunc="size").sort_index(ascending=False), text_auto=True, color_continuous_scale="Blues", title="Customers by recency and frequency score").show()

,Customers,Revenue,MedianR,MedianF,MedianM,CustomerShare,RevenueShare
Segment,,,,,,,
Champions,732,"8,424,689.59",8.00,13.00,"4,489.90",0.13,0.52
Loyal customers,995,"3,409,749.78",29.00,6.00,"1,758.49",0.17,0.21
Cooling regulars,678,"1,894,156.75",94.00,6.00,"1,901.53",0.12,0.12
At-risk high value,618,"1,334,763.15",368.50,4.00,"1,232.92",0.11,0.08
Promising,786,"462,917.46",70.00,2.00,453.82,0.13,0.03
Hibernating,825,"362,565.63",330.00,1.00,327.85,0.14,0.02
Lost one-timers,893,"361,685.77",550.00,1.00,246.75,0.15,0.02
New inflow,312,"87,652.19",26.00,1.00,268.94,0.05,0.01


**Reading this.** Champions are about 12.5% of customers and over half of revenue. At-risk high-value customers have median recency over a year, so this is past spend that could be worth trying to recover, not revenue currently at stake. Recency and frequency scores are correlated, which is why customers cluster along the grid's diagonal.

## 11. Preprocessing for clustering
All three measures are strongly right-skewed. `log1p` compresses that, and `StandardScaler` then gives each mean 0 and variance 1. Without scaling, spend in pounds would dominate days and order counts in any distance calculation. CustomerID is never a feature.

In [5]:
print("Skew before log:", rfm[["Recency", "Frequency", "Monetary"]].skew().round(2).to_dict())
print("Skew after log: ", rfm[["logR", "logF", "logM"]].skew().round(2).to_dict())
print("Scaled means:   ", rfm[["zR", "zF", "zM"]].mean().round(3).to_dict())
print("Scaled std:     ", rfm[["zR", "zF", "zM"]].std().round(3).to_dict())

Skew before log: {'Recency': 0.89, 'Frequency': 11.99, 'Monetary': 26.87}
Skew after log:  {'logR': -0.55, 'logF': 1.01, 'logM': 0.24}
Scaled means:    {'zR': -0.0, 'zF': 0.0, 'zM': -0.0}
Scaled std:      {'zR': 1.0, 'zF': 1.0, 'zM': 1.0}


## 12. Clustering and evaluation

In [6]:
km = pd.DataFrame(evals["kmeans"]); wd = pd.DataFrame(evals["ward"]); db = evals["dbscan"]
display(km.round(3)); display(wd.round(3).T)
print("DBSCAN:", db)
px.line(km, x="k", y="wcss", markers=True, title="Elbow").show()
fig = go.Figure(); fig.add_scatter(x=km.k, y=km.silhouette, name="K-means"); fig.add_scatter(x=wd.k, y=wd.silhouette, name="Ward, sample of 3,000")
fig.update_layout(title="Silhouette"); fig.show()

,k,wcss,silhouette,stability,smallest_share
0,2,"8,544.72",0.44,1.00,0.39
1,3,"6,326.93",0.34,0.99,0.21
2,4,"4,922.24",0.36,0.99,0.20
3,5,"4,101.38",0.34,1.00,0.08
4,6,"3,558.61",0.34,0.97,0.08
5,7,"3,182.52",0.31,0.96,0.06
6,8,"2,894.20",0.30,0.83,0.06
7,9,"2,653.61",0.29,0.94,0.04
8,10,"2,464.03",0.29,0.93,0.04


,0,1,2,3,4,5,6,7,8
k,2.00,3.00,4.00,5.00,6.00,7.00,8.00,9.00,10.00
silhouette,0.40,0.33,0.31,0.29,0.28,0.27,0.28,0.24,0.24


DBSCAN: {'eps': 0.5737905938837875, 'clusters': 1, 'noise_customers': 55, 'noise_revenue_share': 0.2599335689055487, 'noise_median_spend': 38855.24, 'noise_high_spend': 45}


**Choosing k.** The elbow bends gently with no clear point, so it cannot decide alone. Silhouette is highest at k=2, but that only separates active from lapsed customers, which marketing already knows. Among more useful counts, k=4 (0.357) and k=5 (0.343) are nearly the same. **We choose k=5** because results are the same across random starts (adjusted Rand index 1.00), the smallest group is still about 8% of customers, and it splits out a small top tier that makes over half of revenue and needs different treatment. This is a business decision backed by the metrics, and the metrics alone do not force it. Silhouette around 0.34 also means the clusters overlap: customers fall along a continuum, not into sharply separate groups.

**Comparing methods.**

| Method | Result here | Use |
|---|---|---|
| K-means | Stable, interpretable, best silhouette at every k | The segmentation |
| Ward hierarchical | Same broad shape, slightly lower silhouette, sample only | Check on k and structure |
| DBSCAN | One dense mass plus 55 outliers | A list of whale accounts to manage personally, not segments |

## 13. Segment profiling

In [7]:
prof = rfm.groupby("k5").agg(Customers=("CustomerID", "size"), MedianR=("Recency", "median"), MedianF=("Frequency", "median"), MedianM=("Monetary", "median"), Revenue=("Monetary", "sum"))
prof["CustomerShare"] = prof.Customers / prof.Customers.sum(); prof["RevenueShare"] = prof.Revenue / prof.Revenue.sum()
names = {0: "Dormant one-timers", 1: "Recent starters", 2: "Lapsed regulars", 3: "Active regulars", 4: "Elite repeat buyers"}
prof.insert(0, "Name", prof.index.map(names)); display(prof)
display(pd.crosstab(rfm.k5.map(names), rfm.Segment, normalize="index").round(2))
outliers = rfm[rfm.IsNoise].sort_values("Monetary", ascending=False)
display(outliers.head(8)[["CustomerID", "Country", "Recency", "Frequency", "Monetary"]])

,Name,Customers,MedianR,MedianF,MedianM,Revenue,CustomerShare,RevenueShare
k5,,,,,,,,
0,Dormant one-timers,1639,411.00,1.00,229.75,"421,970.40",0.28,0.03
1,Recent starters,1104,26.00,2.00,572.89,"718,535.62",0.19,0.04
2,Lapsed regulars,1366,260.00,3.00,"1,055.76","1,869,566.86",0.23,0.11
3,Active regulars,1274,31.00,8.00,"2,813.29","4,388,324.75",0.22,0.27
4,Elite repeat buyers,456,7.00,23.00,"9,556.17","8,939,782.69",0.08,0.55


Segment,At-risk high value,Champions,Cooling regulars,Hibernating,Lost one-timers,Loyal customers,New inflow,Promising
k5,,,,,,,,
Active regulars,0.03,0.28,0.24,0.00,0.00,0.45,0.00,0.00
Dormant one-timers,0.00,0.00,0.00,0.37,0.48,0.00,0.01,0.13
Elite repeat buyers,0.01,0.80,0.03,0.00,0.00,0.16,0.00,0.00
Lapsed regulars,0.42,0.00,0.25,0.16,0.07,0.01,0.00,0.10
Recent starters,0.00,0.01,0.01,0.00,0.00,0.31,0.27,0.39


,CustomerID,Country,Recency,Frequency,Monetary
5654,18102,United Kingdom,0,145,"578,408.64"
2253,14646,Netherlands,1,144,"522,885.29"
1771,14156,EIRE,9,142,"297,875.18"
2511,14911,EIRE,1,370,"258,630.86"
5014,17450,United Kingdom,8,49,"233,643.91"
1315,13694,United Kingdom,3,141,"190,130.56"
5073,17511,United Kingdom,2,59,"168,728.06"
67,12415,Australia,24,23,"143,107.02"


**Reading this.** Names come from the medians in the table above, so each one can be checked against the numbers. Rule-based segments and clusters mostly agree: Elite repeat buyers are largely Champions, and Dormant one-timers are largely Hibernating and Lost. Two independent methods landing on similar groups gives more confidence in the structure than either would alone.

## 14. Business recommendations

| Group | Evidence | Action | How to measure it |
|---|---|---|---|
| Champions / Elite | About 12.5% of customers, over half of revenue, median 8 days since last order | Loyalty perks, early access, referrals. Avoid blanket discounts | Retention and order frequency against a control group |
| Whale accounts (DBSCAN) | 55 accounts, 26% of revenue | Named account manager and a churn alert when an account goes quiet | Revenue kept among these accounts |
| Loyal customers | About 17% of customers, 21% of revenue | Cross-sell from ranges they have not bought, using the pair list | Items per order |
| Cooling regulars | Median 94 days since last order, against a 25-day reorder gap | Reminder timed to the usual reorder gap | Reactivation rate |
| At-risk high value | £1.33m past spend, median over a year since last order | Direct win-back test with a holdout group, starting with the largest | Reactivated revenue against offer cost |
| New inflow | Recent, one order | Welcome sequence aimed at a second order | Second-order rate within 60 days |
| Hibernating / Lost | About 30% of customers, under 5% of revenue | Low-cost seasonal mail only | Cost per reactivated customer |

Beyond customer groups: commit Q4 stock before September, review dead stock and the long tail, and capture contact details on guest orders. All of these should be tested, because this analysis shows association, not cause.

## 15. Limitations and next steps
- **No cost or margin data.** Revenue is not profit, so a large customer may not be a profitable one.
- **No stock data.** Dead stock can only be inferred from sales.
- **Truncated history.** Customers who bought before Dec 2009 look new at first, and the December 2009 cohort is inflated. December 2011 is partial.
- **Guest orders** (about 13% of sales) are outside RFM. The same buyer could also appear under two accounts.
- **Country** is the customer's most common invoice country, which may be a billing location.
- **Exact duplicates** were removed. Any that were real orders are lost, worth under 0.3% of revenue.
- **RFM is a snapshot** as of 10 Dec 2011, just after the Christmas buying season, which favours autumn buyers on recency.
- **Returns** are netted at the original sale price, and unmatched returns are not netted at all.

Next steps: add margin data, test the win-back and second-order campaigns with holdout groups, model churn and lifetime value, and refresh scores monthly.

## 16. Final conclusion
Once the double-counted week, cancellations and non-product entries are dealt with, this is a £18.9m trade-heavy UK business with a sharp autumn peak. Revenue is concentrated in a fifth of products and a tenth of customers, which makes protecting those accounts and that stock the first priority. RFM and clustering point to the same groups, and the most useful next action is a measured test: retain the top tier, try to recover at-risk high-value customers, and turn more first orders into second ones.